# Attention 设计指标 PyTorch 实验

这个 notebook 配合 `README.md` 和 `attention_metrics_torch.py` 使用。目标不是训练完整 LLM，而是用小实验理解 attention 设计的 proxy 指标：信息流、稳定性、表达力、位置能力和硬件效率。

In [ ]:
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from attention_metrics_torch import (
    make_causal_mask,
    make_sliding_window_mask,
    receptive_field_matrix,
    graph_diameter,
    qk_logit_stats,
    attention_probs,
    normalized_attention_entropy,
    batch_effective_rank,
    head_diversity,
    estimate_kv_cache_bytes,
    benchmark_forward,
    TinyKVModel,
    make_kv_retrieval_batch,
    retrieval_accuracy,
)

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 1. 信息流：Receptive Field 与 Graph Diameter

下面比较 full causal attention 和 sliding-window attention。`receptive_field_matrix(mask, layers)` 会告诉我们：第 `layers` 层的每个输出 token 能收到哪些输入 token 的信息。

In [ ]:
seq_len = 64
layers_list = [1, 2, 4, 8, 16]
window = 8

masks = {
    'full': make_causal_mask(seq_len),
    f'sliding_window_{window}': make_sliding_window_mask(seq_len, window),
}

for name, mask in masks.items():
    d = graph_diameter(mask, max_layers=seq_len)
    print(name, 'diameter =', d['diameter'], 'unreachable_pairs =', d['unreachable_pairs'])

plt.figure(figsize=(8, 4))
for name, mask in masks.items():
    sizes = []
    for layers in layers_list:
        rf = receptive_field_matrix(mask, layers)
        sizes.append(rf[-1].sum().item())
    plt.plot(layers_list, sizes, marker='o', label=name)
plt.xlabel('layers')
plt.ylabel('receptive field size for last token')
plt.title('Receptive field growth')
plt.grid(True)
plt.legend()
plt.show()

解释：full attention 一层就能看到所有历史 token。sliding-window attention 的理论感受野随层数线性增长。如果层数不够，远距离信息即使理论上存在于序列中，也无法传到当前 token。

## 2. 稳定性：QK Logits 与归一化 Entropy

随机生成 Q/K，比较不同 mask 下的 logits 分布和 attention entropy。真实训练时要按 layer/head/position bucket 分开统计。

In [ ]:
B, H, T, D = 4, 8, 128, 64
q = torch.randn(B, H, T, D)
k = torch.randn(B, H, T, D)

for name, mask in {
    'full': make_causal_mask(T),
    'sliding_32': make_sliding_window_mask(T, 32),
}.items():
    stats = qk_logit_stats(q, k, mask)
    probs = attention_probs(q, k, mask)
    ent = normalized_attention_entropy(probs, mask)
    ranks = batch_effective_rank(probs[0])
    div = head_diversity(probs, mask)
    print('\n', name)
    print('logits:', stats)
    print('entropy mean/std:', ent.mean().item(), ent.std(unbiased=False).item())
    print('effective rank mean:', ranks.mean().item())
    print('head diversity:', div)

可靠性提醒：

- `max` 容易被异常值误导，优先看 `p99` / `p99.9`。
- entropy 要用 `H/log(N_visible)`，否则不同位置和不同 mask 不可比。
- head diversity 要去掉 mask baseline，否则 causal mask 会让所有 head 看起来天然相似。

## 3. 硬件：KV Cache 与 Forward Benchmark

KV cache 经常是长上下文推理的瓶颈。GQA/MQA 的收益主要来自减少 `n_kv_heads`。

In [ ]:
layers = 32
batch_size = 8
seq_len = 8192
n_heads = 32
head_dim = 128
dtype_bytes = 2  # bf16/fp16

for label, n_kv_heads in {'MHA': 32, 'GQA': 8, 'MQA': 1}.items():
    gb = estimate_kv_cache_bytes(layers, batch_size, seq_len, n_kv_heads, head_dim, dtype_bytes) / 1024**3
    print(label, f'{gb:.2f} GiB')

In [ ]:
tokens, targets, vocab_size = make_kv_retrieval_batch(
    batch_size=32,
    num_pairs=16,
    num_keys=128,
    num_values=128,
    device=device,
)

model = TinyKVModel(
    vocab_size=vocab_size,
    max_seq_len=tokens.shape[1],
    d_model=128,
    n_heads=4,
    n_layers=2,
    attn_kind='full',
).to(device)

result = benchmark_forward(lambda x: model(x), tokens, warmup=3, iters=10)
result

## 4. 小任务：Key-Value Retrieval

这个任务测精确检索能力：序列里有若干 `key, value` 对，最后给出 query key，模型要输出对应 value。它比完整 LLM 便宜，但能暴露局部 attention 的远距离信息问题。

In [ ]:
def train_retrieval(attn_kind='full', window=None, steps=300, num_pairs=16, lr=3e-4):
    torch.manual_seed(123)
    sample_tokens, _, vocab_size = make_kv_retrieval_batch(8, num_pairs, 256, 256, device=device)
    model = TinyKVModel(
        vocab_size=vocab_size,
        max_seq_len=sample_tokens.shape[1],
        d_model=128,
        n_heads=4,
        n_layers=2,
        attn_kind=attn_kind,
        window=window,
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    losses = []
    for step in range(steps):
        model.train()
        tokens, targets, _ = make_kv_retrieval_batch(64, num_pairs, 256, 256, device=device)
        logits = model(tokens)
        loss = F.cross_entropy(logits, targets)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
        if (step + 1) % 100 == 0:
            val_tokens, val_targets, _ = make_kv_retrieval_batch(512, num_pairs, 256, 256, device=device)
            acc = retrieval_accuracy(model, val_tokens, val_targets)
            print(attn_kind, window, 'step', step + 1, 'loss', f'{loss.item():.3f}', 'acc', f'{acc:.3f}')
    return model, losses

full_model, full_losses = train_retrieval('full', steps=300, num_pairs=16)
local_model, local_losses = train_retrieval('sliding', window=8, steps=300, num_pairs=16)

plt.figure(figsize=(8, 4))
plt.plot(full_losses, label='full')
plt.plot(local_losses, label='sliding window=8')
plt.xlabel('step')
plt.ylabel('cross entropy')
plt.title('Key-value retrieval training loss')
plt.grid(True)
plt.legend()
plt.show()

如果 sliding-window 模型在这个任务上明显更差，不一定说明它不能做 LLM，但说明它的精确远距离检索下限较弱。下一步可以增加层数、window、global token 或 memory token，再重复实验。